# 第 7 周 · 笔记本 4：评估微调模型

## 练习目标

评估 **QLoRA / LoRA 微调后**的价格预测模型，并和基线对比：

1. 从 **Hugging Face Hub** 加载微调后的适配器（adapters）
2. 在测试集上跑推理与指标评估
3. 用图表可视化误差
4. 与基座 Llama、课程里其它模型结果比较

**课程叙事里的预期**：平均误差约 **$40** 量级（基线约 **$110**）

## 和本课第 7 周的关系

| 概念 | 本笔记本里你会看到 |
|------|-------------------|
| 4-bit 量化加载底座 | `BitsAndBytesConfig` + `AutoModelForCausalLM` |
| 只挂 LoRA 适配器 | `PeftModel.from_pretrained(base, FINETUNED_MODEL_ID)` |
| 测试集评估 | `Item.from_hub` + `evaluate(...)` |
| 与基线对比 | Plotly 柱状图 / 全模型榜 |

## 怎么跑

1. 先完成微调（例如 Colab 上的 `03_*.ipynb`），把适配器推到 Hub
2. 把下面单元格里的 `FINETUNED_MODEL_ID` 换成你的仓库 id
3. `.env` 里准备好 `HF_TOKEN`；有 GPU 更佳
4. 从上到下依次运行（预计 15–20 分钟，视 `EVAL_SIZE` 与硬件而定）


In [ ]:
# ========== 导入：评估栈 + Hub 登录 + 项目内工具 ==========

# sys：改模块搜索路径，便于 import 上级目录的 src
import sys
# 把仓库根（本笔记本的上一级）加入 path，才能找到 src.*
sys.path.append('..')

# os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# torch：检测 CUDA、把张量放到 GPU
import torch
# load_dotenv：从 .env 加载密钥，避免把 token 写进代码
from dotenv import load_dotenv
# login：用 Hugging Face token 登录（拉 gated 模型 / 私有适配器）
from huggingface_hub import login
# AutoTokenizer / AutoModelForCausalLM：加载分词器与因果语言模型
# BitsAndBytesConfig：4-bit 量化配置（与训练时一致，省显存）
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# PeftModel：把 LoRA 适配器挂到已量化的底座上
from peft import PeftModel

# Item：课程项目里的商品样本（含 prompt / price 等）
from src.items import Item
# evaluate：在测试集上算平均误差等指标
from src.evaluator import evaluate
# config：数据集名、基座模型、评测规模等集中配置
from src.config import config

# ---------- 环境：密钥进进程 + 登录 Hub ----------
# 加载 .env（不擅自改 override 等参数）
load_dotenv()
# 从环境变量取 Hugging Face token（键名 HF_TOKEN 不可改）
hf_token = os.environ['HF_TOKEN']
# 登录 Hub；add_to_git_credential=True 便于后续 git/hf 凭据复用
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
# 提示当前是否看得到 CUDA GPU
print(f"GPU available: {torch.cuda.is_available()}")


## 配置：要评估的微调模型 id

把 `FINETUNED_MODEL_ID` 换成你推到 Hub 的仓库（与训练笔记本里的 `HUB_MODEL_ID` 一致）。


In [ ]:
# ========== 指定要评估的微调模型（Hub 仓库 id） ==========

# 占位符 your-username/... 请换成你的用户名与仓库名；字符串结构保持原样
FINETUNED_MODEL_ID = "your-username/llama-pricer-lite"  # Replace with your model

print(f"Will evaluate model: {FINETUNED_MODEL_ID}")
# 打印项目 config（数据集、基座、EVAL_SIZE 等），开跑前核对
config.display()


## 加载测试数据

从 Hub 数据集取测试划分（`Item.from_hub` 返回 train / val / test）。


In [ ]:
# ========== 从 Hub 拉测试集 Item 列表 ==========

print(f"Loading test data from: {config.DATASET_NAME}")
# 只要 test；前两个划分用 _ 丢弃
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")


## 加载微调模型

流程：先 4-bit 加载底座 → 再 `PeftModel.from_pretrained` 挂上 LoRA 适配器。


In [ ]:
# ========== 4-bit 底座 + LoRA 适配器 ==========

# nf4 + double quant：常见 QLoRA 推理侧量化配方；计算用 float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
# 分词器与底座 model id 一致
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
# 按量化配置加载因果 LM；device_map="auto" 自动分配设备
base_model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA adapters from: {FINETUNED_MODEL_ID}")
# 在底座上挂载 Hub 上的适配器权重 → 得到可推理的 PeftModel
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_ID)

# ---------- padding：批推理 / generate 需要 pad_token ----------
if tokenizer.pad_token is None:
    # 没有 pad 时借用 eos，避免 generate 报错
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Fine-tuned model loaded")
# 粗估显存占用（字节 → GB）
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 先对少量样本做冒烟测试

确认 `predict_finetuned` 能吐出可读的价格补全，再跑全量评估。


In [ ]:
# ========== 单条推理：prompt → generate → 抽出 completion ==========

def predict_finetuned(item: Item) -> str:
    """
    Predict price using fine-tuned model
    """
    # 若样本还没有 prompt，按配置的 MAX_TOKENS 生成训练/测试用提示
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

    # 测试用 prompt（通常不含答案，留给模型补全）
    prompt = item.test_prompt()

    # 分词并放到模型所在设备（CPU/GPU）
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 推理：关闭梯度，省显存；温度/采样设置保持原样
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    # 把 token id 解码成文本（跳过特殊符号）
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 按课程 PREFIX 切开，取模型补全段（价格相关文本）
    completion = response.split(config.PREFIX)[-1].strip()

    return completion


In [ ]:
# ========== 抽 5 条测试样本，人工扫一眼预测质量 ==========

print("Testing fine-tuned model on 5 sample products:\n")

for i in range(5):
    # 取第 i 条测试 Item
    item = test[i]
    # 调用上面的微调模型预测函数
    prediction = predict_finetuned(item)

    # 标题截断打印，避免刷屏；真实价与模型输出对照
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)


## 在测试集上做完整评估

`evaluate` 会按 `config.EVAL_SIZE` 抽样/截断规模，并用你的 `predict_finetuned` 当预测器。


In [ ]:
# ========== 全量（或配置规模）评估：平均误差 / MSE / R² ==========

# workers=1：GPU 推理时顺序跑更稳（参数保持原样）
results = evaluate(
    predict_finetuned,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("FINE-TUNED MODEL RESULTS")
print(f"{'='*60}")
# average_error：平均绝对误差量级（美元）
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")


## 与基线（未微调底座）比较

柱状图对比 Base Llama 与你的微调模型；`baseline_error` 来自课程基线数字。


In [ ]:
# ========== Plotly：基线误差 vs 微调误差 ==========

import plotly.graph_objects as go

# baseline_error：课程里 Base Llama 的参考 MAE；finetuned 用上格 results
baseline_error = 110.72  # Base Llama
finetuned_error = results['average_error']
# 相对基线的误差下降百分比
improvement = (baseline_error - finetuned_error) / baseline_error * 100

# 新建空图，再加一条柱状 trace
fig = go.Figure()

fig.add_trace(go.Bar(
    x=["Base Llama 3.2", "Fine-tuned Llama"],
    y=[baseline_error, finetuned_error],
    marker_color=["darkred", "green"],
    text=[f"${baseline_error:.2f}", f"${finetuned_error:.2f}"],
    textposition="outside",
))

fig.update_layout(
    title=f"Fine-tuning Improvement: {improvement:.1f}% reduction in error",
    yaxis_title="Mean Absolute Error ($)",
    width=800,
    height=500,
)

fig.show()

print(f"\n🎉 Fine-tuning reduced error by {improvement:.1f}%!")
print(f"   From ${baseline_error:.2f} → ${finetuned_error:.2f}")


## 与课程中全部参考模型比较

横轴是各方法/模型名，纵轴是平均绝对误差（美元）；绿色柱是你的微调结果。


In [ ]:
# ========== 课程榜单 + 你的微调结果 ==========

# 元组：(显示名, 柱颜色, MAE)；最后一项用本笔记本算出的 finetuned_error
all_results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", 110.72),
    ("Fine-tuned (Your Model)", "green", finetuned_error),
]

# 拆成三个并行序列，供 Plotly 使用
labels, colors, values = zip(*all_results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="All Models Comparison - Price Prediction Error",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1200,
    height=600,
)

fig.show()


## 小结

✅ 评估流程跑完！

**把你的数字填进这里（对照输出）：**
- 微调模型平均误差：$XX.XX
- 相对基座改善：XX%
- 与 GPT-5.1 比：更好 / 更差约 $XX

**课程叙事中的预期量级：**
- Lite 模式：约 ~$65 误差（不错）
- Full 模式：约 ~$40 误差（非常好——叙事上可挑战 GPT-5.1）

**你刚完成的关键步骤：**
1. ✅ 用 QLoRA 思路微调过的适配器完成推理加载
2. ✅ 相对基座误差可下降约 60–70%（视训练档位而定）
3. ✅ 在无按次 API 计费的前提下，本地 / 廉价 GPU 可复现
4. ✅ 模型与数据仍由你掌控

**后续可做：**
- 部署到生产（Modal.com、Hugging Face Endpoints 等）
- 尝试多域适配器、思维链等进阶技巧
- 进入第 8 周多代理系统；第 7 + 第 8 周可拼成完整平台
